# Interactive simulation checks: anemia screening

Verifies the IFA effect across the hemoglobin pipelines, anemia-status assignment,
hemoglobin-screening coverage (baseline vs the `anemia_screening_vv` scenario), and
hemoglobin-test sensitivity/specificity. Ported from the research portfolio VnV notebook
`model_18.3_interactive_simulation_anemia_screening`; updated to the current Engine
(`vivarium.engine`) API and converted to assert-based checks.

Note: the source's `raw_hemoglobin.exposure` pipeline no longer exists (dropped here), and
several exploratory ferritin-screening checks that the source flagged as anomalous were
left out pending clarification -- worth adding back once that behavior is confirmed.

In [ ]:
import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

import numpy as np
import pandas as pd
from pathlib import Path

import vivarium_gates_mncnh
from vivarium.artifact import Artifact
from vivarium.engine import InteractiveContext
from vivarium.engine.framework.configuration import build_model_specification

In [ ]:
!pip list | grep vivarium

In [ ]:
SPEC_PATH = Path(vivarium_gates_mncnh.__file__).parent / "model_specifications/model_spec.yaml"
PIPELINES = ["ifa_deleted_hemoglobin.exposure", "hemoglobin.exposure", "first_anc_hemoglobin.exposure"]

def build_sim(scenario=None):
    spec = build_model_specification(SPEC_PATH)
    del spec.configuration.observers
    spec.configuration.population.population_size = 20_000 * 10
    if scenario is not None:
        spec.configuration.intervention.scenario = scenario
    sim = InteractiveContext(spec)
    sim.step()  # step past the pregnancy timestep
    return sim

def screening_frame(sim):
    pop = sim.get_population()
    values = pd.concat([sim.get_value(p)(pop.index) for p in PIPELINES], axis=1)
    return pd.concat([pop, values], axis=1)

def anemia_status_from(hb):
    return np.where(hb <= 70, "severe",
           np.where(hb <= 100, "moderate",
           np.where(hb <= 110, "mild", "not_anemic")))

In [ ]:
# Baseline scenario
sim = build_sim()
df = screening_frame(sim)
df[["anc_attendance", "oral_iron_intervention", "hemoglobin_screening_coverage",
    "tested_hemoglobin", "anemia_status_during_pregnancy"]].head()

## IFA effect is applied to the right hemoglobin pipelines

In [ ]:
# `hemoglobin.exposure` gets the IFA effect for everyone on IFA; `first_anc_hemoglobin`
# gets it only for first-trimester ANC attendees on IFA. Effect size is deterministic,
# so the deltas are 0 or a single positive constant.
ifa = df.oral_iron_intervention == "ifa"
first_tri = df.anc_attendance.isin(["first_trimester_only", "first_trimester_and_later_pregnancy"])
hb_delta = df["hemoglobin.exposure"] - df["ifa_deleted_hemoglobin.exposure"]
fa_delta = df["first_anc_hemoglobin.exposure"] - df["ifa_deleted_hemoglobin.exposure"]

assert (hb_delta[ifa] > 1e-6).all() and (hb_delta[~ifa].abs() < 1e-6).all(), \
    "IFA effect on hemoglobin.exposure not applied exactly to the IFA-treated"
assert ((fa_delta > 1e-6) == (ifa & first_tri)).all(), \
    "first_anc_hemoglobin IFA effect not restricted to first-trimester ANC attendees on IFA"

## Anemia status matches the hemoglobin thresholds

In [ ]:
# anemia_status_during_pregnancy is assigned from the ifa-deleted hemoglobin thresholds
# (severe <=70, moderate <=100, mild <=110). NOTE: researchers flagged that this should
# ultimately use first_anc_hemoglobin (see vivarium_gates_mncnh PR #149); this asserts
# current behavior.
computed = anemia_status_from(df["ifa_deleted_hemoglobin.exposure"])
tested_mask = df.anemia_status_during_pregnancy != "N/A"
assert (df.anemia_status_during_pregnancy[tested_mask].values == computed[tested_mask]).all(), \
    "anemia status inconsistent with ifa-deleted hemoglobin thresholds"

## Screening coverage: baseline

In [ ]:
# No screening without ANC; partial hemoglobin screening at ANC; no ferritin screening at baseline.
cov = df.groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert cov.loc["none"] == 0, "hemoglobin screening occurred among no-ANC simulants at baseline"
attended = cov.drop("none")
assert (attended > 0).all() and (attended < 1).all(), \
    f"expected partial baseline hemoglobin screening at ANC, got {attended.to_dict()}"
assert (~df.ferritin_screening_coverage).all(), "ferritin screening should be off at baseline"

## Hemoglobin test sensitivity / specificity

In [ ]:
# Among screened simulants: sensitivity P(test low | truly low) ~= 0.85,
# specificity P(test adequate | truly adequate) ~= 0.80 (truth = ifa-deleted hemoglobin < 100).
tested = df[df.tested_hemoglobin != "not_tested"].copy()
tested["truth"] = np.where(tested["ifa_deleted_hemoglobin.exposure"] < 100, "low", "adequate")
sens = (tested.loc[tested.truth == "low", "tested_hemoglobin"] == "low").mean()
spec = (tested.loc[tested.truth == "adequate", "tested_hemoglobin"] == "adequate").mean()
assert np.isclose(sens, 0.85, atol=0.05), f"hemoglobin-test sensitivity {sens:.3f} far from 0.85"
assert np.isclose(spec, 0.80, atol=0.05), f"hemoglobin-test specificity {spec:.3f} far from 0.80"

## Screening coverage: `anemia_screening_vv` scale-up scenario

In [ ]:
# In the anemia-screening VnV scenario, every ANC attendee is screened for hemoglobin and
# no-ANC simulants are still never screened.
vv = build_sim(scenario="anemia_screening_vv")
vv_pop = vv.get_population()
vv_cov = vv_pop.groupby("anc_attendance").hemoglobin_screening_coverage.mean()
assert vv_cov.loc["none"] == 0, "hemoglobin screening among no-ANC simulants in anemia_screening_vv"
assert (vv_cov.drop("none") == 1).all(), \
    f"expected 100% hemoglobin screening at ANC in anemia_screening_vv, got {vv_cov.to_dict()}"